In [ ]:
!pip install -q FlagEmbedding

In [ ]:
from google.colab import files
files.upload()   # het_goods.parquet файлын таңдаңыз

Saving het_goods.parquet to het_goods.parquet


{'het_goods.parquet': b'PAR1\x15\x04\x15\xc0\xda\x06\x15\xce\x87\x04L\x15\xa8k\x15\x00\x12\x00\x00\xa0\xad\x03(\'\xcc-\x03\x00\x00\x00\x00\x837*\x05\x08\x082g+\x05\x08\x04<7\t\x18\x046\xc5\t\x08\x08\xca\x9d?\x05\x18\x04\xe0\xbc\t \x04\xc1"\t\x18\x04\xee\x1f\t\x10\x08\xb1\x11/\x05 \x04\xd2V\t\x18\x04W\xcb\tP\x04\x1d\x08\t\x10\x00\x12\r\x18\x04\x94\xfe\t\x10\x049r\t\x08\x08\xf9\x86,\x058\x04\xf8C\t\x08\x08S\xaf5\t\x10\x00V\tX\x04\x8f$\tX\x04\x05\xaf\t \x04t\xfd\t\x08\x04\xbc\x8f\t\x08\x04\x81\x12\tH\x08\x91\x97>\x058\x00%\rH\x005\r\x08\x00\x82\r \x04~\x14\t(\x08x\xc14\x05(\x04\x1a\xde\t`\x04\xb9K\tH\x00\xc0\r\x08\x08\x83\x16.\x05 \x04\x1e\xbb\tx\x00\x82\r\x10\x04\xaeg\t(\x04\x1aL\tH\x08\x9f\xff1\x05(\x04C\xa1\tH\x04D\xf0\t \x08\xad\xa8)\x05\x18\x04\x07\x03)\x00\x08w(0\x05\x10\x04K\x8e\t(\x04hI\t\x08\x04j\xf3\t\x18\x00i\r\x08\x04\xb9\xd2\t0\x04\xf0"\tH\x04rd\t(\x04(\x99\t\x10\x04!\xed\t\x08\x00\xe3-H\x00V\r`\x04)a\t(\x04L\x8b\t@\x04S\x8f\t\x08\x04\x1a\x8d\t\x08\x04\x99:\t\x08\x04\xe39\tp\

In [ ]:
import numpy as np, pandas as pd, torch
from FlagEmbedding import BGEM3FlagModel

df = pd.read_parquet("het_goods.parquet")
df = df[(df["unit_price_st"] > 0) & df["plan_extra_desc"].notna()].reset_index(drop=True)
df["lp"] = np.log(df["unit_price_st"])
texts = df["plan_extra_desc"].astype(str).tolist()

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
out = model.encode(texts, batch_size=64, max_length=256,
                   return_dense=True, return_sparse=True, return_colbert_vecs=True)

dense = out["dense_vecs"]          # (n, 1024), нормаланған
lex = out["lexical_weights"]       # әр мәтінге {токен: салмақ}
colb = out["colbert_vecs"]         # әр мәтінге (токендер, 1024)
print(len(texts), dense.shape)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 108/108 [00:03<00:00, 31.80it/s]

6868 (6868, 1024)


In [ ]:
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

# Сөз салмақтарын сирек матрицаға жинау
r, c, v = [], [], []
for i, d in enumerate(lex):
    for tok, w in d.items():
        r.append(i); c.append(int(tok)); v.append(float(w))
X_sp = csr_matrix((v, (r, c)), shape=(len(lex), max(c) + 1))

X_tf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                       sublinear_tf=True).fit_transform(texts)

def colbert_sim(idx, chunk=8):
    vecs = [torch.as_tensor(colb[i], dtype=torch.float16, device="cuda") for i in idx]
    n, L = len(vecs), max(x.shape[0] for x in vecs)
    P = torch.zeros(n, L, vecs[0].shape[1], dtype=torch.float16, device="cuda")
    mask = torch.zeros(n, L, dtype=torch.bool, device="cuda")
    for k, x in enumerate(vecs):
        P[k, :len(x)] = x
        mask[k, :len(x)] = True
    S = torch.empty(n, n, device="cuda")
    for a in range(0, n, chunk):
        sim = torch.einsum("qld,pmd->qplm", P[a:a + chunk], P)            # MaxSim
        sim = sim.masked_fill(~mask[None, :, None, :], -1e4)
        best = sim.max(-1).values.masked_fill(~mask[a:a + chunk, None, :], 0)
        S[a:a + chunk] = best.float().sum(-1) / mask[a:a + chunk].sum(-1, keepdim=True)
    return S.cpu().numpy()

rng = np.random.default_rng(0)
methods = {
    "random":  lambda idx: rng.random((len(idx), len(idx))),
    "tfidf":   lambda idx: (X_tf[idx] @ X_tf[idx].T).toarray(),
    "dense":   lambda idx: dense[idx] @ dense[idx].T,
    "sparse":  lambda idx: (X_sp[idx] @ X_sp[idx].T).toarray(),
    "colbert": colbert_sim,
}

In [ ]:
def evaluate(sim_fn, ks=(3, 5, 10,20,30,40,50,60,70,80,130,160,230), min_cust=2):
    res = {k: ([], []) for k in ks}
    for _, g in df.groupby("gk"):
        if len(g) < 4 or g["customer_bin"].nunique() < 2:
            continue
        idx = g.index.to_numpy()
        S = sim_fn(idx)
        cust, lp = g["customer_bin"].to_numpy(), g["lp"].to_numpy()
        for a in range(len(idx)):
            other = np.where(cust != cust[a])[0]
            order = other[np.argsort(-S[a, other])]
            for k in ks:
                top = order[:k]
                if len(top) < k or len(set(cust[top])) < min_cust:
                    continue
                res[k][0].append(abs(lp[a] - np.median(lp[top])))
                res[k][1].append(abs(lp[a] - np.median(lp[other])))
    return {k: (len(m), np.exp(np.median(m)), np.exp(np.median(c))) for k, (m, c) in res.items()}

for name, fn in methods.items():
    for k, (n, m, c) in evaluate(fn).items():
        print(f"{name:<8} k={k:<3} {n:>6} жол | әдіс {m:.2f} есе | код {c:.2f} есе")

random   k=3     5745 жол | әдіс 3.41 есе | код 2.93 есе
random   k=5     5823 жол | әдіс 3.27 есе | код 2.94 есе
random   k=10    5493 жол | әдіс 3.08 есе | код 2.95 есе
random   k=20    4724 жол | әдіс 2.98 есе | код 2.93 есе
random   k=30    4157 жол | әдіс 2.88 есе | код 2.92 есе
random   k=40    3519 жол | әдіс 2.85 есе | код 2.83 есе
random   k=50    3122 жол | әдіс 2.84 есе | код 2.83 есе
random   k=60    2782 жол | әдіс 2.79 есе | код 2.74 есе
random   k=70    2580 жол | әдіс 2.70 есе | код 2.71 есе
random   k=80    2244 жол | әдіс 2.75 есе | код 2.75 есе
random   k=130   1348 жол | әдіс 2.48 есе | код 2.48 есе
random   k=160   1113 жол | әдіс 2.46 есе | код 2.46 есе
random   k=230    805 жол | әдіс 2.16 есе | код 2.18 есе
tfidf    k=3     4368 жол | әдіс 3.03 есе | код 2.99 есе
tfidf    k=5     5208 жол | әдіс 2.99 есе | код 3.04 есе
tfidf    k=10    5314 жол | әдіс 2.97 есе | код 3.01 есе
tfidf    k=20    4686 жол | әдіс 2.89 есе | код 2.94 есе
tfidf    k=30    4152 жол | әді

In [ ]:
rng = np.random.default_rng(0)
M = 20   # барлық gk үшін бірдей таңдама көлемі

def spread_fixed(lp, m=M, reps=20):
    vals = []
    for _ in range(reps):
        s = rng.choice(lp, m, replace=False)
        vals.append(np.median(np.abs(s - np.median(s))))
    return np.exp(np.median(vals))

rows = []
for gk, g in df.groupby("gk"):
    if len(g) < 30 or g["customer_bin"].nunique() < 3:
        continue
    lp = g["lp"].to_numpy()
    rows.append({"gk": gk, "n": len(g), "n_cust": g["customer_bin"].nunique(),
                 "spread_all": np.exp(np.median(np.abs(lp - np.median(lp)))),
                 "spread_20": spread_fixed(lp)})
st = pd.DataFrame(rows)
st["cat"] = st["gk"].str[:2]
st["size"] = pd.cut(st["n"], [29, 50, 100, 200, 100_000],
                    labels=["30–50", "50–100", "100–200", "200+"])

# 1) Көлем топтары бойынша: барлық жол мен бірдей 20 жол
print(st.groupby("size", observed=True)
        .agg(gk=("n", "size"), spread_all=("spread_all", "median"),
             spread_20=("spread_20", "median")).round(2), "\n")

# 2) Корреляциялар: жалпы және санат ішінде
st["l20"] = np.log(st["spread_20"])
for col in ("n", "n_cust"):
    st["x"] = np.log(st[col])
    within = (st["x"] - st.groupby("cat")["x"].transform("median")).corr(
        st["l20"] - st.groupby("cat")["l20"].transform("median"), method="spearman")
    print(f"{col:<7} жалпы: {st['x'].corr(st['l20'], method='spearman'):+.2f} | "
          f"санат ішінде: {within:+.2f}")

         gk  spread_all  spread_20
size                              
30–50    27        2.37       2.21
50–100   18        3.01       2.77
100–200  10        2.79       2.61
200+      2        3.58       3.23 

n       жалпы: +0.17 | санат ішінде: +0.15
n_cust  жалпы: -0.07 | санат ішінде: -0.18


In [ ]:
from sklearn.cluster import AgglomerativeClustering

SIM = 0.85   # бір топқа кіру үшін косинустық ұқсастық шегі
df["cl"] = ""
for gk, g in df.groupby("gk"):
    idx = g.index.to_numpy()
    if len(idx) == 1:
        df.loc[idx, "cl"] = gk + "#0"
        continue
    lab = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average",
                                  distance_threshold=1 - SIM
                                  ).fit_predict(dense[idx].astype(np.float32))
    df.loc[idx, "cl"] = [f"{gk}#{l}" for l in lab]

print(f"gk: {df['gk'].nunique()} | топтар: {df['cl'].nunique()}")

gk: 400 | топтар: 3300


In [ ]:
def tight_segments(df, key, cut="2026-01-01", min_n=5, min_cust=3, max_spread=1.3):
    tr = df#[df["sign_date"] < cut]
    te = df#[df["sign_date"] >= cut]

    agg = tr.groupby(key).agg(n=("lp", "size"), n_cust=("customer_bin", "nunique"),
                              med=("lp", "median"))
    agg["spread"] = np.exp(tr.groupby(key)["lp"].apply(
        lambda x: np.median(np.abs(x - x.median()))))
    ok = agg[(agg["n"] >= min_n) & (agg["n_cust"] >= min_cust)]
    tight = ok[ok["spread"] <= max_spread]

    # 2026 жолдарын 2024–2025 медианасымен салыстырамыз
    te = te[te[key].isin(ok.index)].copy()
    te["dev"] = (te["lp"] - te[key].map(ok["med"])).abs()
    te["tight"] = te[key].isin(tight.index)

    print(f"--- {key} ---")
    print(f"Бағаланатын сегменттер: {len(ok)} | тығыздары: {len(tight)}")
    print(te.groupby("tight")["dev"].agg(rows="size",
                                         spread=lambda d: np.exp(d.median())).round(2))
    # print(f"2026 жолдарының тығыз сегменттерге түсетін үлесі: "
    #       f"{te['tight'].sum() / (df['sign_date'] >= cut).sum():.1%}\n")
    return tight

tight_gk = tight_segments(df, "gk")
tight_cl = tight_segments(df, "cl")

# Тығыз топтардың мысалдары
ex = (df[df["cl"].isin(tight_cl.index)]
      .groupby("cl").agg(n=("lp", "size"), price=("unit_price_st", "median"),
                         desc=("plan_extra_desc", "first")))
ex["desc"] = ex["desc"].str[:70]
print(ex.sort_values("n", ascending=False).head(15))

--- gk ---
Бағаланатын сегменттер: 167 | тығыздары: 10
       rows  spread
tight              
False  5984    2.54
True    111    1.21
--- cl ---
Бағаланатын сегменттер: 67 | тығыздары: 13
       rows  spread
tight              
False   622    2.21
True    108    1.12
                            n         price  \
cl                                            
611011.200.000000|5114#2   17  4.169872e+06   
639910.000.000000|5114#12  13  7.891495e+04   
331219.206.000000|5114#0   12  2.364627e+06   
620920.000.000013|5114#4   12  2.745518e+05   
749020.000.000009|5114#2    8  5.564764e+06   
749020.000.000009|5114#4    8  4.151515e+06   
639910.000.000000|5114#4    7  2.118766e+05   
351210.900.000000|5114#10   6  1.026389e+08   
331910.800.000000|5114#2    5  1.089241e+06   
620230.000.000001|5114#19   5  1.598685e+06   
620920.000.000013|5114#5    5  7.617254e+05   
639910.000.000000|5114#0    5  8.161960e+04   
692010.000.000000|5114#0    5  1.164478e+06   

                         